# Fase 3 — Validación y Comparativa de modelos

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook:
1. Loads all saved predictions from Fase 2
2. Aligns them on the **same test-set timestamps**
3. Computes a unified metrics table (MAPE, RMSE, MAE, training time)
4. Produces publication-quality comparison plots
5. Optionally compares against the REE official forecast (ESIOS indicator 544)

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_theme(style='whitegrid')
print('Environment ready.')

## 1 · Load saved predictions

In [ ]:
def load_preds(path):
    df = pd.read_csv(path, parse_dates=['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)
    return df

pred_files = {
    'SARIMAX'       : 'data/predictions_sarimax.csv',
    'LSTM'          : 'data/predictions_lstm.csv',
    'TTM zero-shot' : 'data/predictions_ttm_zeroshot.csv',
    'TTM few-shot'  : 'data/predictions_ttm_fewshot.csv',
}

preds = {}
for name, path in pred_files.items():
    if os.path.exists(path):
        preds[name] = load_preds(path)
        print(f'{name:<18}: {len(preds[name]):,} rows  ({preds[name]["datetime"].min().date()} → {preds[name]["datetime"].max().date()})')
    else:
        print(f'{name:<18}: NOT FOUND — run the corresponding fase2 notebook first')

In [ ]:
# Load metrics JSON files
metric_files = {
    'SARIMAX'       : 'data/metrics_sarimax.json',
    'LSTM'          : 'data/metrics_lstm.json',
    'TTM zero-shot' : 'data/metrics_ttm_zeroshot.json',
    'TTM few-shot'  : 'data/metrics_ttm_fewshot.json',
}

all_metrics = {}
for name, path in metric_files.items():
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[name] = json.load(f)
        print(f'{name}: {all_metrics[name]}')
    else:
        print(f'{name}: metrics file NOT FOUND')

## 2 · Align on a common time window

Different models may cover slightly different test ranges (due to stride differences).
We intersect on the common timestamps.

In [ ]:
if len(preds) > 1:
    # Find common timestamps across all loaded predictions
    common_dates = None
    for name, df in preds.items():
        ts = set(df['datetime'])
        common_dates = ts if common_dates is None else common_dates & ts

    common_dates = sorted(common_dates)
    print(f'Common timestamps: {len(common_dates):,}')

    # Filter each prediction DataFrame
    preds_aligned = {}
    for name, df in preds.items():
        preds_aligned[name] = df[df['datetime'].isin(common_dates)].set_index('datetime').sort_index()

    # Single ground-truth series (same across all models)
    y_true = preds_aligned[list(preds_aligned.keys())[0]]['y_true'].values
    dates  = preds_aligned[list(preds_aligned.keys())[0]].index

    print(f'Aligned test window: {dates.min().date()} → {dates.max().date()}')
else:
    print('Only one model loaded — skipping alignment.')
    preds_aligned = preds

## 3 · Recompute metrics on aligned window

In [ ]:
aligned_metrics = {}

for name, df in preds_aligned.items():
    m = du.compute_metrics(df['y_true'].values, df['y_pred'].values, label=name)
    # Carry over timing from saved JSON if available
    if name in all_metrics:
        m['train_s']     = all_metrics[name].get('train_s', 0)
        m['inference_s'] = all_metrics[name].get('inference_s', 0)
    aligned_metrics[name] = m

## 4 · Comparison table

In [ ]:
summary_df = du.plot_comparison_table(
    aligned_metrics,
    save_path = 'data/fig_comparison_table.png',
)
summary_df.to_csv('data/comparison_table.csv')
print('\nSaved → data/comparison_table.csv')

## 5 · Visual comparison

In [ ]:
# Choose a representative week for plotting
PLOT_DAYS   = 7
PLOT_START  = dates[0]
PLOT_END    = PLOT_START + pd.Timedelta(days=PLOT_DAYS)

model_colors = {
    'SARIMAX'      : '#e377c2',
    'LSTM'         : '#d62728',
    'TTM zero-shot': '#2ca02c',
    'TTM few-shot' : '#17becf',
}

fig, ax = plt.subplots(figsize=(15, 5))

mask = (dates >= PLOT_START) & (dates < PLOT_END)

ax.plot(dates[mask], y_true[mask], label='Actual', color='#1f77b4', linewidth=1.8, zorder=10)

for name, df in preds_aligned.items():
    y = df['y_pred'].values
    ax.plot(dates[mask], y[mask], label=name,
            color=model_colors.get(name, 'gray'), linewidth=1.2, linestyle='--', alpha=0.9)

ax.set_title(f'All models — actual vs predicted ({PLOT_START.date()} to {PLOT_END.date()})')
ax.set_ylabel('Demand (MW)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %Hh'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('data/fig_all_models_week.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# MAPE bar chart across models
names  = list(aligned_metrics.keys())
mapes  = [aligned_metrics[n]['mape'] for n in names]
rmses  = [aligned_metrics[n]['rmse'] for n in names]
maes   = [aligned_metrics[n]['mae']  for n in names]

x = np.arange(len(names))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, vals, ylabel, title in [
    (axes[0], mapes, 'MAPE (%)',  'MAPE by model'),
    (axes[1], rmses, 'RMSE (MW)', 'RMSE by model'),
    (axes[2], maes,  'MAE (MW)',  'MAE by model'),
]:
    bars = ax.bar(x, vals, color=[model_colors.get(n, '#7f7f7f') for n in names], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('data/fig_metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plot: actual vs predicted for all models (sample 2000 points)
N_SCATTER = 2000
n_cols    = len(preds_aligned)
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5), sharey=True)

if n_cols == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, preds_aligned.items()):
    idx_sample = np.random.choice(len(df), size=min(N_SCATTER, len(df)), replace=False)
    yt = df['y_true'].values[idx_sample]
    yp = df['y_pred'].values[idx_sample]
    ax.scatter(yt, yp, alpha=0.2, s=5, color=model_colors.get(name, 'gray'))
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel('Actual (MW)')
    m = aligned_metrics[name]
    ax.text(0.05, 0.95, f"MAPE={m['mape']:.2f}%\nRMSE={m['rmse']:.0f}MW",
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[0].set_ylabel('Predicted (MW)')
plt.suptitle('Actual vs Predicted — all models', y=1.01)
plt.tight_layout()
plt.savefig('data/fig_scatter_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training time comparison
train_times = {
    n: aligned_metrics[n].get('train_s', 0)
    for n in aligned_metrics
    if aligned_metrics[n].get('train_s', 0) > 0
}

if train_times:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(
        list(train_times.keys()),
        [v / 60 for v in train_times.values()],
        color=[model_colors.get(n, 'gray') for n in train_times],
        alpha=0.85,
    )
    ax.set_xlabel('Training time (minutes)')
    ax.set_title('Training time per model')
    plt.tight_layout()
    plt.savefig('data/fig_training_time.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6 · (Optional) REE official forecast benchmark

Compare all models against the REE programmed demand forecast (ESIOS indicator 544).
Uncomment and run only if you have a valid ESIOS API key.

In [ ]:
# OPTIONAL — uncomment to run
# 
# test_start = str(dates.min().date())
# test_end   = str(dates.max().date())
# 
# df_ree = du.download_esios(
#     indicator_id = du.ESIOS_REE_FORECAST,
#     start_date   = test_start,
#     end_date     = test_end,
# )
# df_ree = df_ree.rename(columns={'value': 'y_pred'}).set_index('datetime')
# df_ree['y_true'] = preds_aligned[list(preds_aligned.keys())[0]]['y_true']
# df_ree = df_ree.dropna()
# 
# m_ree = du.compute_metrics(df_ree['y_true'].values, df_ree['y_pred'].values, label='REE forecast')
# aligned_metrics['REE forecast'] = m_ree
# print('REE benchmark added.')

## 7 · Final summary table

In [ ]:
rows = []
for name, m in aligned_metrics.items():
    rows.append({
        'Model'          : name,
        'MAPE (%)'       : round(m['mape'],  3),
        'RMSE (MW)'      : round(m['rmse'],  1),
        'MAE (MW)'       : round(m['mae'],   1),
        'Train time (s)' : round(m.get('train_s', 0), 0),
        'Inference (s)'  : round(m.get('inference_s', 0), 2),
    })

final_table = pd.DataFrame(rows).set_index('Model')
print(final_table.to_string())
final_table.to_csv('data/final_results_table.csv')
print('\nSaved → data/final_results_table.csv')

## Summary

Fill in after running all notebooks:

| Model | MAPE (%) | RMSE (MW) | MAE (MW) | Train (min) |
|-------|----------|-----------|----------|-------------|
| SARIMAX        | … | … | … | … |
| LSTM           | … | … | … | … |
| TTM zero-shot  | … | … | … |  0 |
| TTM few-shot   | … | … | … | … |

**Best model:** …  
**Conclusions:** …